# 🚀 Hancock Cybersecurity LLM Training

**Dataset**: 4,951 cybersecurity training records  
**Model**: TinyLlama-1.1B-Chat-v1.0  
**Method**: QLoRA 4-bit fine-tuning  
**GPU**: T4 (15GB VRAM)  
**Time**: ~30-45 minutes  

---

## ⚡ Quick Start

1. **Runtime → Change runtime type → T4 GPU**
2. **Upload** `unified-expanded.jsonl` when prompted
3. **Run all cells** (Runtime → Run all)
4. **Download** trained model from `/content/hancock-v1/`

## 📦 Install Dependencies

In [ ]:
%%capture
!pip install -q transformers peft bitsandbytes accelerate datasets sentencepiece

## 📤 Upload Dataset

Upload `data/hancock/unified-expanded.jsonl` from your local machine:

In [ ]:
from google.colab import files
import os

print("📤 Upload your dataset file (unified-expanded.jsonl)")
uploaded = files.upload()

# Rename to expected path
if uploaded:
    uploaded_file = list(uploaded.keys())[0]
    os.rename(uploaded_file, 'unified-expanded.jsonl')
    print(f"✅ Dataset uploaded: {uploaded_file}")
    
    # Verify
    !wc -l unified-expanded.jsonl
    !du -h unified-expanded.jsonl

## 🧠 Training Script (GPU-Optimized)

In [ ]:
import json
import torch
from pathlib import Path
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

# ========== Configuration ==========
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_PATH = "unified-expanded.jsonl"
OUTPUT_DIR = "/content/hancock-v1"
MAX_LENGTH = 512
BATCH_SIZE = 4  # GPU can handle larger batches
GRADIENT_ACCUMULATION = 4
EPOCHS = 3
LEARNING_RATE = 2e-4

print("🔧 Configuration:")
print(f"   Model: {MODEL_NAME}")
print(f"   Dataset: {DATASET_PATH}")
print(f"   Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * GRADIENT_ACCUMULATION})")
print(f"   Epochs: {EPOCHS}")
print(f"   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"   CUDA available: {torch.cuda.is_available()}")

## 📊 Load Dataset

In [ ]:
def load_jsonl_dataset(path: str) -> Dataset:
    """Load JSONL dataset and format for training"""
    print(f"📂 Loading dataset from {path}...")
    
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if line.strip():
                try:
                    record = json.loads(line)
                    # Format as instruction-following
                    text = f"### Instruction:\n{record['instruction']}\n\n### Response:\n{record['output']}"
                    records.append({"text": text})
                except json.JSONDecodeError as e:
                    print(f"⚠️  Skipping line {line_num}: {e}")
    
    print(f"✅ Loaded {len(records):,} records")
    return Dataset.from_list(records)

dataset = load_jsonl_dataset(DATASET_PATH)

## 🤖 Load Model with QLoRA

In [ ]:
print("🤖 Loading model and tokenizer...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ Model loaded with QLoRA")

## 🔤 Tokenize Dataset

In [ ]:
def preprocess_function(examples):
    """Tokenize examples"""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

print("🔤 Tokenizing dataset...")
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset.column_names
)
print(f"✅ Tokenized {len(tokenized_dataset):,} examples")

## 🏋️ Train Model

In [ ]:
# Training arguments (GPU-optimized)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=True,  # GPU supports FP16
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    warmup_steps=100,
    report_to="none"
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# Train!
print("\n🚀 Starting training...")
print(f"   Total steps: {len(tokenized_dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION) * EPOCHS}")
print(f"   Estimated time: 30-45 minutes\n")

trainer.train()

print("\n✅ Training complete!")

## 💾 Save Model

In [ ]:
print("💾 Saving model...")

# Save model and tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save training summary
summary = {
    "model": MODEL_NAME,
    "dataset": DATASET_PATH,
    "records": len(dataset),
    "epochs": EPOCHS,
    "output_dir": OUTPUT_DIR,
    "status": "complete"
}

with open(f"{OUTPUT_DIR}/training_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print(f"✅ Model saved to: {OUTPUT_DIR}")
print(f"   Files:")
!ls -lh {OUTPUT_DIR}

## 📥 Download Trained Model

Zip and download the trained model:

In [ ]:
!zip -r hancock-v1.zip {OUTPUT_DIR}

from google.colab import files
files.download('hancock-v1.zip')

print("\n✅ Download started!")
print(f"   Size: {!du -h hancock-v1.zip}")

## 🧪 Test Model (Optional)

In [ ]:
print("🧪 Testing trained model...\n")

# Test prompts
test_prompts = [
    "Explain SQL injection vulnerability",
    "What is ASLR and how does it improve security?",
    "How do I use nmap for network scanning?"
]

for prompt in test_prompts:
    inputs = tokenizer(f"### Instruction:\n{prompt}\n\n### Response:\n", return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"❓ {prompt}")
    print(f"💬 {response.split('### Response:')[1].strip()}")
    print("-" * 80)

print("\n✅ Testing complete!")

---

## 🎯 Success!

Your Hancock Cybersecurity LLM is now trained!

**Next Steps**:
1. Download `hancock-v1.zip`
2. Extract to `/tmp/peachtree/models/hancock-v1/`
3. Test locally or deploy to production

**Model Capabilities**:
- Cybersecurity vulnerability analysis
- Penetration testing guidance
- Security tool usage (nmap, sqlmap, Metasploit)
- Bug bounty research
- Blockchain security auditing
- System hardening recommendations

**Training Stats**:
- Dataset: 4,951 cybersecurity records
- Model: TinyLlama-1.1B + QLoRA (2.25M trainable params)
- Training time: ~30-45 minutes on T4 GPU
- Memory: ~4GB VRAM (4-bit quantization)

---

**Created by**: Trainer Handoff Agent  
**Project**: PeachTree Dataset Control Plane  
**Organization**: 0ai-Cyberviser  
**Date**: April 28, 2026